# GH Sequence Space — Week 2 Module

Part A: CAZy family download -> length/redundancy inspection -> MMseqs2 clustering -> Sequence Similarity Network (SSN) ->

Part B: Structural modelling (Chai-1).

This notebook starts with **Step 1: downloading a CAZy family**. Set `GH_FAMILY` below to whichever family you want to explore (e.g. `GH26`).

## 1. Fetch CAZy family accession list

In [ ]:
# @title CAZy family to explore
# =======================================================================
# Config -- set the family here, everything downstream reuses this
# =======================================================================
GH_FAMILY = "GH26" # @param {type:"string"}

import pandas as pd
from pathlib import Path

work_dir = Path(GH_FAMILY)
work_dir.mkdir(exist_ok=True)

GH_FAMILY

## Fetch Entry Data from CAZy

In [ ]:
# @title Download Data from CAZy or Use Uploaded File
# Please select ONE option below. If both are selected, 'Use Uploaded CAZy File' will be prioritized.
Download_from_CAZy = True # @param {type:"boolean"}
Upload_CAZy_File = False # @param {"type":"boolean"}

import pandas as pd
from pathlib import Path
import requests
from google.colab import files # Import google.colab.files for upload functionality

cazy_url = f"https://www.cazy.org/IMG/cazy_data/{GH_FAMILY}.txt"
standard_cazy_file_path = work_dir / f"{GH_FAMILY}.txt"

loaded_cazy_df = pd.DataFrame()
data_loaded_successfully = False

# --- Determine action based on user selection ---
if Upload_CAZy_File:
    # Prioritize loading an uploaded file if selected
    if standard_cazy_file_path.exists():
        print(f"Loading data from existing local file: {standard_cazy_file_path}")
        try:
            loaded_cazy_df = pd.read_csv(
                standard_cazy_file_path,
                header=None,
                sep="\t",
                names=["family", "kingdom", "organism", "genbank_accession", "source"],
            )
            data_loaded_successfully = True
            print(f"Successfully loaded {len(loaded_cazy_df)} entries from local file.")
        except Exception as e:
            print(f"Error loading data from {standard_cazy_file_path}: {e}")
            print("Please ensure the local file is correctly formatted or re-upload it.")
    else:
        print(f"'{GH_FAMILY}.txt' not found locally. Please upload the file now.")
        try:
            uploaded = files.upload()
            if uploaded:
                for filename, content in uploaded.items():
                    # Save the uploaded file with the standard name
                    with open(standard_cazy_file_path, 'wb') as f:
                        f.write(content)
                    print(f"File '{filename}' uploaded successfully and saved as '{standard_cazy_file_path}'.")
                    break # Assuming only one file is relevant
                # After successful upload, try to load it
                if standard_cazy_file_path.exists():
                    try:
                        loaded_cazy_df = pd.read_csv(
                            standard_cazy_file_path,
                            header=None,
                            sep="\t",
                            names=["family", "kingdom", "organism", "genbank_accession", "source"],
                        )
                        data_loaded_successfully = True
                        print(f"Successfully loaded {len(loaded_cazy_df)} entries from uploaded file.")
                    except Exception as e:
                        print(f"Error loading data from uploaded file {standard_cazy_file_path}: {e}")
                        print("Please ensure the uploaded file is correctly formatted.")
            else:
                print("No file was uploaded. Please upload the file or select 'Download from CAZy'.")
        except Exception as e:
            print(f"Error during file upload: {e}")

elif Download_from_CAZy:
    # Attempt to download if 'Download_from_CAZy' is selected and 'Upload_CAZy_File' is False
    if standard_cazy_file_path.exists():
        print(f"Local file {standard_cazy_file_path} already exists. Skipping download and loading it.")
        try:
            loaded_cazy_df = pd.read_csv(
                standard_cazy_file_path,
                header=None,
                sep="\t",
                names=["family", "kingdom", "organism", "genbank_accession", "source"],
            )
            data_loaded_successfully = True
            print(f"Successfully loaded {len(loaded_cazy_df)} entries from existing local file.")
        except Exception as e:
            print(f"Error loading data from {standard_cazy_file_path}: {e}")
            print("Please ensure the existing local file is correctly formatted or delete it to force re-download.")
    else:
        print(f"Attempting to download {GH_FAMILY}.txt from {cazy_url} using requests...")
        try:
            response = requests.get(cazy_url, timeout=30)
            response.raise_for_status()
            with open(standard_cazy_file_path, 'wb') as f:
                f.write(response.content)
            print("Download complete.")
            loaded_cazy_df = pd.read_csv(
                standard_cazy_file_path,
                header=None,
                sep="\t",
                names=["family", "kingdom", "organism", "genbank_accession", "source"],
            )
            data_loaded_successfully = True
            print(f"Successfully loaded {len(loaded_cazy_df)} entries from downloaded file.")
        except requests.exceptions.RequestException as e:
            print(f"Error downloading file from {cazy_url}: {e}")
            print("Please check the URL, your internet connection, or try again later.")
            print(f"Consider uploading '{GH_FAMILY}.txt' using the upload cell and selecting 'Upload CAZy File'.")
        except Exception as e:
            print(f"Error loading downloaded data from {standard_cazy_file_path}: {e}")

else: # Neither option is selected
    print("No data retrieval method selected. Please select either 'Download from CAZy' or 'Upload CAZy File'.")
    print("0 entries loaded. cazy_df is empty.")

# --- Final Assignment and Display ---
cazy_df = loaded_cazy_df

if data_loaded_successfully:
    print(f"\nFinal cazy_df contains {len(cazy_df)} entries for {GH_FAMILY}.")
    display(cazy_df.head())
    display(cazy_df.tail())
else:
    print("\nNo data loaded into cazy_df. It remains empty.")
    display(cazy_df.head())
    display(cazy_df.tail())

## Fetch HTML with characterized GH26s

In [ ]:
# @title Download the characterised GH26 list from the course repo
# CAZy's listing of characterised GH26 entries is kept in the course repository,
# so there is nothing to upload by hand.
!wget -q -O GH26_characterized.html https://raw.githubusercontent.com/UadKLab/27200_Data-Driven-Bioengineering/main/weeks/week02_sequence_structure_and_ptms/data/GH26_characterized.html
!ls -lh GH26_characterized.html

In [ ]:
# @title ####Parse Characterized Members (from HTML to CSV)
import pandas as pd
from pathlib import Path

html_file_name = "GH26_characterized.html"
html_file_path = Path(html_file_name)

if not html_file_path.exists() or html_file_path.stat().st_size == 0:
    raise FileNotFoundError(
        f"'{html_file_name}' is missing or empty. Run the download cell above first."
    )

tables = pd.read_html(html_file_path)
main_table = tables[1]

col_names = ["Protein Name", "EC#", "Reference", "Organism", "GenBank", "Uniprot", "PDB/3D"]
main_table = main_table.iloc[:, :7].copy()   # <-- .copy() added here
main_table.columns = col_names

divider_mask = main_table.apply(lambda r: r.nunique() == 1, axis=1)
kingdom_rows = main_table[divider_mask]["Protein Name"]

main_table["Kingdom"] = None
for idx, kingdom in kingdom_rows.items():
    main_table.loc[idx, "Kingdom"] = kingdom
main_table["Kingdom"] = main_table["Kingdom"].ffill()

main_table = main_table[~divider_mask].copy()   # <-- .copy() added here too
main_table = main_table[main_table["Protein Name"] != "Protein Name"]
main_table = main_table[["Kingdom", "Protein Name", "EC#", "Reference", "Organism", "GenBank", "Uniprot", "PDB/3D"]].reset_index(drop=True)

main_table.to_csv("GH26_characterized.csv", index=False)

print("Content of GH26_characterized.csv:")
display(pd.read_csv("GH26_characterized.csv"))

In [ ]:
# @title Heatmap of Sequences by Source and Kingdom
import matplotlib.pyplot as plt
import seaborn as sns

# Group by source and kingdom, then count occurrences
taxon_counts = cazy_df.groupby(['source', 'kingdom']).size().unstack(fill_value=0)

# Create the heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(taxon_counts, annot=True, fmt='d', cmap='viridis', linewidths=.5)
plt.title(f'Heatmap of {GH_FAMILY} Sequences by Source and Kingdom')
plt.xlabel('Taxonomic_Kingdom')
plt.ylabel('Source_Database')
plt.tight_layout()
plt.show()

This heatmap visualizes the number of sequences for each combination of `source database` (NCBI or JGI) and `Taxonomic_kingdom`. The numbers in the cells indicate the count of sequences, and the color intensity represents higher counts.

# **QUESTIONS**
### How many entries are in this CAZy family ?

### What is the origin of the sequences ?

### How many characterized enzymes ?



---

# **2. Fetch sequences from NCBI**
## Since fetching sequences from JGI is not trivial, we will now fetch sequence from NCBI



In [ ]:
# @title Split NCBI and JGI entries
# @title
import pandas as pd

# Define the output file paths
ncbi_output_path = work_dir / f"{GH_FAMILY}_ncbi.txt"
jgi_output_path = work_dir / f"{GH_FAMILY}_jgi.txt"

# Filter the DataFrame for 'ncbi' source and save
cazy_df[cazy_df['source'] == 'ncbi'].to_csv(ncbi_output_path, sep='\t', index=False, header=False)
print(f"Saved NCBI-sourced entries to {ncbi_output_path}")

# Filter the DataFrame for 'jgi' source and save
cazy_df[cazy_df['source'] == 'jgi'].to_csv(jgi_output_path, sep='\t', index=False, header=False)
print(f"Saved JGI-sourced entries to {jgi_output_path}")

# Display the first few lines of the newly created files to verify
print("\nFirst 5 lines of GH26_ncbi.txt:")
!head -n 5 {ncbi_output_path}

print("\nFirst 5 lines of GH26_jgi.txt:")
!head -n 5 {jgi_output_path}

## Fetch sequences from NCBI

In [ ]:
# @title Prepare NCBI Accessions Dataframe
# Load accessions from GH26_ncbi.txt
ncbi_file_path = work_dir / f"{GH_FAMILY}_ncbi.txt"

# Read the file, assuming tab-separated and accession is the 4th column (index 3)
ncbi_df = pd.read_csv(ncbi_file_path, sep='\t', header=None,
                       names=['family', 'kingdom', 'organism', 'genbank_accession', 'source',
                              'genus', 'TaxID_Genus_x', 'Superkingdom_Genus_x', 'Phylum_Genus_x',
                              'Class_Genus_x', 'Order_Genus_x', 'Family_Genus_x', 'ScientificName_Genus_x',
                              'TaxID_Genus_y', 'Superkingdom_Genus_y', 'Phylum_Genus_y',
                              'Class_Genus_y', 'Order_Genus_y', 'Family_Genus_y', 'ScientificName_Genus_y', 'Phylum_Genus_Clean'])

accessions = ncbi_df['genbank_accession'].dropna().tolist()
print(f"Loaded {len(accessions)} accessions from {ncbi_file_path}")

In [ ]:
# @title Install Biopython
!pip install -q biopython

In [ ]:
# @title Fetch NCBI Sequences in Batches
from Bio import Entrez, SeqIO
import time
from IPython.display import display, Markdown # Import display and Markdown

def fetch_sequences_in_batches(accessions_list, batch_size, entrez_email, entrez_api_key=None):
    """
    Fetches protein sequences from NCBI using Entrez in batches.

    Args:
        accessions_list (list): A list of GenBank accession numbers.
        batch_size (int): The number of accessions to fetch in each batch.
        entrez_email (str): Your email address for Entrez.
        entrez_api_key (str, optional): Your NCBI API key. Defaults to None.

    Returns:
        list: A list of Biopython SeqRecord objects.
    """
    Entrez.email = entrez_email
    if entrez_api_key:
        Entrez.api_key = entrez_api_key

    records = []
    total_accessions = len(accessions_list)

    for i in range(0, total_accessions, batch_size):
        batch = accessions_list[i:i + batch_size]
        print(f"Fetching {i}-{min(i + batch_size, total_accessions)} of {total_accessions}...")
        try:
            handle = Entrez.efetch(
                db="protein",
                id=",".join(batch),
                rettype="fasta",
                retmode="text",
            )
            batch_records = list(SeqIO.parse(handle, "fasta"))
            records.extend(batch_records)
            handle.close()
        except Exception as e:
            print(f"  Batch failed ({e}), retrying individually...")
            for acc in batch:
                try:
                    handle = Entrez.efetch(db="protein", id=acc, rettype="fasta", retmode="text")
                    records.extend(list(SeqIO.parse(handle, "fasta")))
                    handle.close()
                except Exception as e2:
                    print(f"    Failed: {acc} ({e2})")
        time.sleep(0.4)  # stay under NCBI's rate limit

    print(f"\nRetrieved {len(records)} sequences out of {total_accessions} accessions")
    return records

# Set your Entrez email (REQUIRED)
my_entrez_email = "your.email@example.com"  # <-- set your email here
# Set your NCBI API key (OPTIONAL, uncomment if you have one)
# my_entrez_api_key = "your_ncbi_api_key"

# Define batch size
BATCH_SIZE = 200

# Call the function to fetch sequences
records = fetch_sequences_in_batches(
    accessions,
    BATCH_SIZE,
    my_entrez_email,
    # entrez_api_key=my_entrez_api_key # Uncomment if using an API key
)

# Display the summary using IPython.display.Markdown for proper rendering
display(Markdown("""
### Summary of NCBI Sequence Fetching

This cell defines and executes the `fetch_sequences_in_batches` function
to retrieve protein sequences from NCBI. Accessions were successfully fetched.
"""))

In [ ]:
# @title Save Fetched Sequences to FASTA
# Save the fetched sequences to a multi-FASTA file
output_fasta_path = work_dir / f"{GH_FAMILY}_ncbi.fasta"

with open(output_fasta_path, "w") as output_handle:
    SeqIO.write(records, output_handle, "fasta")

print(f"Saved {len(records)} sequences to {output_fasta_path}")

In [ ]:
# @title Display GH26_ncbi.fasta
!head -n 35 GH26/GH26_ncbi.fasta

In [ ]:
# @title Download and install seqkit and view fasta stats
# Download and install seqkit
import os

if not os.path.exists('./seqkit'):
    print("seqkit not found, downloading...")
    !wget -q https://github.com/shenwei356/seqkit/releases/download/v2.13.0/seqkit_linux_amd64.tar.gz
    !tar xzf seqkit_linux_amd64.tar.gz
    !chmod +x seqkit
    print("seqkit downloaded and installed.")
else:
    print("seqkit already installed.")

!./seqkit stats {GH_FAMILY}/{GH_FAMILY}_ncbi.fasta

In [ ]:
# @title Remove duplicate sequences
from Bio import SeqIO # Ensure SeqIO is imported for this cell

input_fasta_original = work_dir / f"{GH_FAMILY}_ncbi.fasta"
deduplicated_fasta = work_dir / f"{GH_FAMILY}_ncbi_deduplicated.fasta"

print(f"Removing duplicate sequences from {input_fasta_original} using Bio.SeqIO and saving to {deduplicated_fasta}...")

unique_records = []
seen_sequences = set() # Use a set to keep track of seen sequence strings

# Read all sequences from the original FASTA file
original_records = list(SeqIO.parse(input_fasta_original, "fasta"))

for record in original_records:
    sequence_str = str(record.seq)
    if sequence_str not in seen_sequences:
        seen_sequences.add(sequence_str)
        unique_records.append(record)

# Write the unique sequences to the deduplicated FASTA file
with open(deduplicated_fasta, "w") as output_handle:
    SeqIO.write(unique_records, output_handle, "fasta")

print(f"Removed {len(original_records) - len(unique_records)} duplicated records.")
print(f"'{deduplicated_fasta}' now contains {len(unique_records)} unique sequences.")

# Update the global 'records' variable for subsequent analysis to use the deduplicated set
records = unique_records

input_fasta = deduplicated_fasta # Update input_fasta to point to the deduplicated file

print(f"\nSummary statistics for {input_fasta} (after deduplication):")
!./seqkit stats {input_fasta}


In [ ]:
# @title Reload Deduplicated Sequences
# Re-load records from the deduplicated FASTA file to ensure subsequent analysis uses this set
from Bio import SeqIO

print(f"Re-loading sequences from {deduplicated_fasta} into 'records' variable...")
records = list(SeqIO.parse(deduplicated_fasta, "fasta"))
print(f"'records' now contains {len(records)} sequences.")

In [ ]:
# @title Analyze Sequence Length Distribution
import matplotlib.pyplot as plt
import numpy as np

# Extract sequence lengths from the records
lengths = [len(str(record.seq)) for record in records]

# Define a directory for saving plots
cluster_dir = work_dir # Using work_dir for now, can be changed later if needed

# Histogram for raw sequence length distribution
plt.figure(figsize=(9, 4))
plt.hist(lengths, bins=60, color='skyblue', edgecolor='black')
plt.xlabel("Sequence length (aa)")
plt.ylabel("Number of sequences")
plt.title(f"{GH_FAMILY} raw sequence length distribution (n={len(records)})\n")
plt.tight_layout()
plt.savefig(cluster_dir / "raw_length_distribution.png", dpi=150)
plt.show()

# Zoomed view of the bulk of the distribution (drop the extreme tail)
p1, p99 = np.percentile(lengths, [1, 99])
plt.figure(figsize=(9, 4))
plt.hist([l for l in lengths if p1 <= l <= p99], bins=60, color='lightgreen', edgecolor='black')
plt.xlabel("Sequence length (aa)")
plt.ylabel("Number of sequences")
plt.title(f"{GH_FAMILY} length distribution, 1st-99th percentile ({p1:.0f}-{p99:.0f} aa)\n")
plt.tight_layout()
plt.savefig(cluster_dir / "raw_length_distribution_zoomed.png", dpi=150)
plt.show()

print(f"\n1st percentile: {p1:.0f} aa, 99th percentile: {p99:.0f} aa")
print("Use these plots to choose your own min/max length cutoff below.")

# **QUESTIONS**
### How many sequences did you fetch ?

### How many duplicate sequences ?

### What is the average sequence length ?



---


# **3. Cluster with MMseqs2**
### Now we well cluster the sequence with MMseqs2 using easy-cluster and easy-search

## Cluster with MMseqs2

In [ ]:
# @title Install and Initialize MMseqs2
!wget https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
!tar xzf mmseqs-linux-avx2.tar.gz

import os
os.environ["PATH"] += ":" + os.path.abspath("mmseqs/bin")

!mmseqs

## MMseqs2 Clustering and Bar Chart Visualization

In [ ]:
# @title MMseqs2 Clustering and Bar Chart Visualization - (top_n = Number of clusters to plot)
### 4. MMseqs2 Clustering and Bar Chart Visualization
!pip install -q biopython
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO # Import SeqIO for FASTA handling

# Define temporary directory for MMseqs2
tmp_dir = work_dir / "mmseqs_tmp"
tmp_dir.mkdir(exist_ok=True)

# =======================================================================
# 4. MMseqs2 clustering at 35% identity (matches original workflow)
# =======================================================================
MIN_SEQ_ID = 0.35 # @param {type:"number"}
COVERAGE = 0.8 # @param {type:"number"}
MIN_LENGTH = 200 # @param {type:"number"}
MAX_LENGTH = 800 # @param {type:"number"}
top_n = 30 # @param {type:"number"} # Allow user to specify how many top clusters to visualize
INCLUDE_ACTIVITIES_COLORING = True # @param {type:"boolean"}
VERBOSE_MODE = False # @param {type:"boolean"}

# input_fasta and cluster_dir are already defined in previous cells
# input_fasta = work_dir / f"{GH_FAMILY}_ncbi.fasta"
# cluster_dir = work_dir # Set in cell 615126f7

# --- Apply sequence length filtering ---
# 'records' variable is expected to contain the deduplicated sequences from previous steps
if 'records' in globals() and records:
    initial_count = len(records)
    filtered_records = [rec for rec in records if MIN_LENGTH <= len(rec.seq) <= MAX_LENGTH]
    filtered_count = len(filtered_records)
    print(f"Filtering sequences by length: {initial_count} sequences reduced to {filtered_count} (min={MIN_LENGTH}, max={MAX_LENGTH} aa).")

    # Save the length-filtered sequences to a new FASTA file
    length_filtered_fasta = work_dir / f"{GH_FAMILY}_ncbi_deduplicated_length_filtered.fasta"
    with open(length_filtered_fasta, "w") as output_handle:
        SeqIO.write(filtered_records, output_handle, "fasta")

    input_fasta = length_filtered_fasta # Update input_fasta to use the filtered file
else:
    print("Warning: 'records' variable not found or empty. Skipping length filtering. Ensure previous cells were run.")
    print(f"Proceeding with input_fasta: {input_fasta}")


print(f"Running MMseqs2 easy-cluster with min-seq-id={MIN_SEQ_ID}, coverage={COVERAGE}...")

verbose_flag = "-v 3" if VERBOSE_MODE else "-v 0"
!mmseqs easy-cluster {input_fasta} {cluster_dir}/{GH_FAMILY.lower()}_cluster {tmp_dir} \
    --min-seq-id {MIN_SEQ_ID} -c {COVERAGE} --cov-mode 0 {verbose_flag}

cluster_tsv = cluster_dir / f"{GH_FAMILY.lower()}_cluster_cluster.tsv"

# Check if the cluster_tsv file exists before trying to read it
if cluster_tsv.exists():
    clusters = pd.read_csv(cluster_tsv, sep="\t", names=["representative", "member"])

    n_sequences = clusters["member"].nunique()
    n_clusters = clusters["representative"].nunique()
    print(f"\n{n_sequences} sequences -> {n_clusters} clusters at {MIN_SEQ_ID*100:.0f}% identity")

    cluster_sizes = clusters.groupby("representative").size().sort_values(ascending=False)
    n_large = (cluster_sizes > 10).sum()
    print(f"{n_large} clusters have >10 sequences")
    print("\nLargest clusters:")
    print(cluster_sizes.head(10))




    # =======================================================================
    # 5. Cluster size bar chart (matches "plotting no. of seqs in each
    #    cluster" slide)
    # =======================================================================

    # Ensure main_table is loaded if not already in global scope
    try:
        if 'main_table' not in globals() or main_table.empty:
            main_table = pd.read_csv("GH26_characterized.csv")
            print("Loaded GH26_characterized.csv for coloring data.")
    except FileNotFoundError:
        print("Warning: GH26_characterized.csv not found. Cannot color by characterized members.")
        main_table = pd.DataFrame() # Create empty DataFrame to avoid errors

    plt.figure(figsize=(16, 8))

    colors = [] # Initialize as empty list
    cmap = None
    norm = None
    cluster_char_counts_for_plot = pd.Series() # Initialize for potential colorbar

    if INCLUDE_ACTIVITIES_COLORING and not main_table.empty:
        characterized_genbanks = set(main_table['GenBank'].dropna().tolist())

        # Create a DataFrame that links every member to its representative and indicates if it's characterized
        all_members_with_reps = clusters.copy()
        all_members_with_reps['is_characterized'] = all_members_with_reps['member'].isin(characterized_genbanks)

        # Group by representative and sum the 'is_characterized' boolean (True=1, False=0)
        characterized_counts_per_rep = all_members_with_reps.groupby('representative')['is_characterized'].sum()

        # Filter for the top_n representatives being plotted
        plotting_rep_ids = cluster_sizes.head(top_n).index
        cluster_char_counts_for_plot = characterized_counts_per_rep.reindex(plotting_rep_ids, fill_value=0)

        max_char_count = cluster_char_counts_for_plot.max() if not cluster_char_counts_for_plot.empty else 0

        if max_char_count > 0:
            cmap = plt.colormaps['rainbow']
            norm = plt.Normalize(vmin=0, vmax=max_char_count)
            for count in cluster_char_counts_for_plot:
                if count == 0:
                    colors.append('black') # Color black if zero characterized members
                else:
                    colors.append(cmap(norm(count)))
        elif not cluster_char_counts_for_plot.empty and max_char_count == 0:
            # All top clusters have zero characterized members, color all black
            print("All top clusters have zero characterized members. Coloring bars black.")
            colors = ['black'] * top_n
        else:
            # This case means cluster_char_counts_for_plot was empty or had no relevant data
            print("No characterized data available for the top clusters. Using default blue.")
            colors = ['#1f77b4'] * top_n
    else:
        # Default blue if INCLUDE_ACTIVITIES_COLORING is False or main_table is empty
        colors = ['#1f77b4'] * top_n

    ax = cluster_sizes.head(top_n).plot(kind="bar", color=colors)
    plt.xlabel("Cluster representative")
    plt.ylabel("Number of sequences")
    plt.title(f"Top {top_n} clusters by size ({GH_FAMILY}, {MIN_SEQ_ID*100:.0f}% identity)")
    plt.xticks(rotation=45, ha='right') # Rotate labels for better readability

    # Add numerical labels on top of each bar
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', label_type='edge', padding=3)

    # Add color bar if activities coloring is enabled and there's data to color by
    if INCLUDE_ACTIVITIES_COLORING and cmap is not None and norm is not None and not cluster_char_counts_for_plot.empty and cluster_char_counts_for_plot.max() > 0:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array(cluster_char_counts_for_plot.values) # Important to set values for the colorbar
        cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.02, pad=0.04)
        cbar.set_label('Number of Characterized Members')

    plt.tight_layout()
    plt.savefig(cluster_dir / "top_clusters_barplot.png", dpi=150)
    plt.show()
else:
    print(f"Error: Cluster file not found at {cluster_tsv}. MMseqs2 clustering might have failed.")


# **QUESTIONS**
### 1. How many sequences have you clustered ?
### 2. How many clusters do they form ?
### 3. How many clusters have more than 10 seqs ?
### 4. Do you see any large uncharacterized clusters ?



---


# Now we can view which GH26 entry represent each cluster

In [ ]:
# @title Sort Representative Sequences by Cluster Size and Save
from Bio import SeqIO

# Path to the FASTA file containing representative sequences
representative_fasta_path = cluster_dir / f"{GH_FAMILY.lower()}_cluster_rep_seq.fasta"

# Load the representative sequences
if representative_fasta_path.exists():
    representative_sequences = list(SeqIO.parse(representative_fasta_path, "fasta"))
    print(f"Loaded {len(representative_sequences)} representative sequences from {representative_fasta_path}")

    # Define the output path for the saved representative sequences
    output_representative_fasta = work_dir / f"{GH_FAMILY}_cluster_representatives.fasta"

    # Save them to a new FASTA file
    with open(output_representative_fasta, "w") as output_handle:
        SeqIO.write(representative_sequences, output_handle, "fasta")

    print(f"Saved representative sequences to {output_representative_fasta}")
else:
    print(f"Error: Representative sequences FASTA file not found at {representative_fasta_path}")

# Get the ordered list of representative IDs from cluster_sizes (already sorted by size)
sorted_rep_ids = cluster_sizes.index.tolist()

# Create a dictionary for quick lookup of SeqRecord objects by their ID
# We'll use the .id attribute for matching
rep_seq_dict = {seq_record.id.split()[0]: seq_record for seq_record in representative_sequences}

# Reorder the representative sequences based on sorted_rep_ids
sorted_representative_sequences = []
for rep_id in sorted_rep_ids:
    if rep_id in rep_seq_dict:
        sorted_representative_sequences.append(rep_seq_dict[rep_id])
    else:
        print(f"Warning: Representative ID {rep_id} not found in loaded sequences.")

# Overwrite the existing FASTA file with the sorted sequences
with open(output_representative_fasta, "w") as output_handle:
    SeqIO.write(sorted_representative_sequences, output_handle, "fasta")

print(f"Successfully sorted representative sequences and saved to {output_representative_fasta}")

In [ ]:
# @title View FASTA Entries for representative from largest clusters

import Bio.SeqIO
from pathlib import Path # Ensure Path is imported

num_fasta_entries = 20 # @param {type:"number"}

# Ensure output_representative_fasta is defined (from cell ff1c29d1)
if 'output_representative_fasta' not in globals():
    print("Warning: 'output_representative_fasta' is not defined. Please ensure the clustering and sorting cells are run.")
    # Attempt to derive it if possible, or set a placeholder
    output_representative_fasta = Path(GH_FAMILY) / f"{GH_FAMILY}_cluster_representatives.fasta"
    if not output_representative_fasta.exists():
        print(f"Error: Representative FASTA file not found at expected path: {output_representative_fasta}")
        output_representative_fasta = None # Indicate it's not available


if output_representative_fasta and output_representative_fasta.exists():
    if num_fasta_entries > 0:
        print(f"Displaying first {num_fasta_entries} intact entries from {output_representative_fasta}:")
        count = 0
        with open(output_representative_fasta, "r") as f:
            for record in Bio.SeqIO.parse(f, "fasta"):
                # Print the full FASTA entry (ID + description + sequence)
                print(f">{record.id} {record.description.replace(record.id, '').strip()}") # Print ID and description
                print(str(record.seq)) # Print the sequence
                count += 1
                if count >= num_fasta_entries:
                    break
        if count == 0:
            print(f"No entries found in {output_representative_fasta}.")
    else:
        print("Please enter a positive number for 'Number of FASTA entries to display'.")
else:
    if output_representative_fasta: # Path was constructed but file doesn't exist
        print(f"The representative FASTA file does not exist at {output_representative_fasta}. Please ensure the previous steps creating this file have been executed successfully.")
    else: # output_representative_fasta was never properly defined
        print("Could not determine the path to the representative FASTA file.")

## Get Members for a Specific Cluster Representative

Use the input field below to specify a cluster representative ID, and the cell will display all the member sequences belonging to that cluster.

In [ ]:
# @title Display FASTA Entries grouped to a Cluster Representative
import pandas as pd
import Bio.SeqIO # Import SeqIO to handle FASTA records
from pathlib import Path # Ensure Path is imported for file paths

# Ensure sorted_rep_ids is available from previous cells (for potential future dropdown use, though currently manual input)
if 'sorted_rep_ids' not in globals():
    print("Warning: 'sorted_rep_ids' is not defined. Please ensure MMseqs2 clustering and sorting cells are run.")
    sorted_rep_ids = []

# Set a default representative ID, though it's an empty string for manual input
default_representative_id = ""

# @title Enter a cluster representative ID to view its members
cluster_representative_id = "BCI51186.1" # @param {type:"string", allow-input:true}

if 'clusters' in globals() and not clusters.empty:
    # Filter the clusters DataFrame for the selected representative
    if cluster_representative_id:
        members_in_cluster = clusters[clusters['representative'] == cluster_representative_id]['member'].tolist()

        if members_in_cluster:
            print(f"Intact FASTA entries for members of cluster represented by '{cluster_representative_id}':")

            # Path to the deduplicated FASTA file (assuming it's named GH_FAMILY_ncbi_deduplicated.fasta)
            # Ensure GH_FAMILY and work_dir are defined from previous cells
            if 'GH_FAMILY' not in globals() or 'work_dir' not in globals():
                print("Error: GH_FAMILY or work_dir is not defined. Cannot load sequences.")
                fasta_file_path = None
            else:
                fasta_file_path = Path(work_dir) / f"{GH_FAMILY}_ncbi_deduplicated.fasta"

            if fasta_file_path and fasta_file_path.exists():
                # Load all sequences into a dictionary for efficient lookup
                all_sequences = SeqIO.to_dict(SeqIO.parse(fasta_file_path, "fasta"))

                found_count = 0
                for member_id in members_in_cluster:
                    if member_id in all_sequences:
                        record = all_sequences[member_id]
                        # Print the full FASTA entry (ID + description + sequence)
                        print(f">{record.id} {record.description.replace(record.id, '').strip()}")
                        print(str(record.seq))
                        found_count += 1
                    else:
                        print(f"Warning: Member '{member_id}' not found in the FASTA file.")
                print(f"Total intact FASTA entries displayed: {found_count} out of {len(members_in_cluster)} members.")
            else:
                print(f"Error: FASTA file not found at {fasta_file_path}. Please ensure sequence fetching and deduplication steps are run.")

        else:
            print(f"No members found for cluster representative '{cluster_representative_id}'.")
            print("Please ensure the ID is correct and is a cluster representative.")
    else:
        print("No cluster representative selected. Please enter one.")
else:
    print("The 'clusters' DataFrame is not available. Please run the MMseqs2 clustering cell first.")



---


# **4. Sequence Similarity Network (SSN)**

A Sequence Similarity Network (SSN) is a powerful tool for visualizing relationships between sequences.
It requires an all-vs-all pairwise search to determine edge weights based on sequence similarity.

First, we'll install `networkx` for graph manipulation and plotting.

In [ ]:
# @title Install NetworkX (needed for SSN visualization)
!pip install -q networkx

Next, we perform an all-vs-all pairwise search using MMseqs2 in order to build an SSN on the full dataset

In [ ]:
# @title all-vs-all pairwise search using MMseqs2 - (you can apply filters if needed)
import subprocess
import os

MIN_SSN_SEQ_ID = 0 # @param {type:"number"} Minimum sequence identity for SSN edges
SSN_COVERAGE = 0 # @param {type:"number"} Minimum coverage for SSN edges

# The full deduplicated sequences FASTA file (around 4000 sequences)
# `input_fasta` was defined earlier in the notebook as work_dir / f"{GH_FAMILY}_ncbi_deduplicated.fasta"
ssn_input_fasta = input_fasta

# Define directories for SSN related files
ssn_dir = work_dir / "ssn"
ssn_dir.mkdir(exist_ok=True)
ssn_tmp = ssn_dir / "tmp"
ssn_tmp.mkdir(exist_ok=True)

allvall_tsv_path = ssn_dir / "allvall.tsv"

# Check if the output file already exists
if not allvall_tsv_path.exists():
    print(f"Running mmseqs easy-search on {ssn_input_fasta} against itself...")

    # Execute mmseqs easy-search command
    try:
        command = [
            "mmseqs", "easy-search",
            ssn_input_fasta.as_posix(),  # Query database
            ssn_input_fasta.as_posix(),  # Target database
            allvall_tsv_path.as_posix(), # Output TSV
            ssn_tmp.as_posix(),    # Temporary directory
            "--format-output", "query,target,pident,alnlen,evalue,bits",
            "-s", str(MIN_SSN_SEQ_ID),          # Sensitivity, now controlled by param
            "-c", str(SSN_COVERAGE) ,
            "--cov-mode", "0",
            "--max-seqs", "1000"  # Max number of sequences to retrieve per query
        ]
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        print("All-vs-all search complete.")
        print(f"Results saved to {allvall_tsv_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error running mmseqs easy-search: {e}")
        print(f"Stdout: {e.stdout}")
        print(f"Stderr: {e.stderr}")
    except FileNotFoundError:
        print("Error: mmseqs command not found. Ensure MMseqs2 is installed and in your PATH.")
else:
    print(f"All-vs-all search results already found at {allvall_tsv_path}. Skipping re-run.")
    print("If you wish to re-run the search, delete the file manually.")

Now we load the pairwise search results and prepare them for network construction. We will remove self-hits as they are not informative for network edges.

In [ ]:
# @title Load Pairwise Search Results
import pandas as pd
import networkx as nx

# Load the pairwise search results
allvall_tsv_path = ssn_dir / "allvall.tsv"
pairs = pd.read_csv(
    allvall_tsv_path,
    sep="\t",
    names=["query", "target", "pident", "alnlen", "evalue", "bits"],
)

# Remove self-hits (where query is the same as target)
pairs = pairs[pairs["query"] != pairs["target"]]
print(f"{len(pairs)} pairwise hits (self-hits removed)")
display(pairs.head())
display(pairs.tail())

## Choose SSN edge thresholds.

This is the crucial step for defining the network. You can adjust `MIN_PIDENT` (minimum percentage identity) and `MAX_EVALUE` (maximum E-value) and `MIN_BITS` (minimum bitscore) to control the stringency of the connections (edges) in your network.

Experimenting with these values is key to exploring the sequence space and identifying subfamilies.

In [ ]:
# @title Choose SSN edge thresholds. You can set minimum percent identity (MIN_PIDENT), maximum E-value (MAX_EVALUE), and minimum bit score (MIN_BITS)
MIN_PIDENT = 25 # @param {type:"number"} # minimum % identity to draw an edge
MAX_EVALUE = 1e-100 # @param ["1e-10","1e-15","1e-20","1e-25","1e-30","1e-35","1e-40","1e-45","1e-50","1e-55","1e-60","1e-65","1e-70","1e-75","1e-80","1e-85","1e-90","1e-95","1e-100","1e-105","1e-110","1e-115","1e-120","1e-125","1e-130","1e-135","1e-140","1e-145","1e-150","1e-155","1e-160","1e-165","1e-170","1e-175","1e-180","1e-185","1e-190","1e-195","1e-200"] {"type":"raw"}
MIN_BITS = 100 # @param {type:"number"} # minimum bit score to draw an edge

# Filter edges based on the chosen thresholds
edges = pairs[(pairs["pident"] >= MIN_PIDENT) & (pairs["evalue"] <= MAX_EVALUE) & (pairs["bits"] >= MIN_BITS)]
print(f"{len(edges)} edges pass threshold (pident >= {MIN_PIDENT}%, evalue <= {MAX_EVALUE}, bits >= {MIN_BITS})")

# Build the NetworkX graph
G = nx.Graph()
for _, row in edges.iterrows():
    G.add_edge(row["query"], row["target"], weight=row["bits"], pident=row["pident"])

# Report network statistics
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
      f"{nx.number_connected_components(G)} connected components")

### Exploring Different Thresholds

This is the core exploratory exercise with SSNs. There isn't a single 'correct' threshold for `MIN_PIDENT`, `MAX_EVALUE` and `MIN_BITS`. You should try different combinations and rerun the cell **Choose an edge threshold** and build the graph' onwards to observe how the network resolves or merges into different connected components (clusters).

This helps in understanding the evolutionary relationships within the protein family.

## SSN Visualization

Now we can visualize the SSN using `matplotlib` with a force-directed layout. Nodes will be colored by their connected component, offering a rough visualization of potential subfamily groupings. To better visualize individual clusters we arrange them in a grid-like fashion.

(Default: number of columns: 15, Spacing factors: 1, base cell unit: 5, scaling 0.5)

In [ ]:
# @title SSN arranged in rows and colored by characterized members
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx

# --- Configuration for grid arrangement ---
NUM_COLUMNS = 15 # @param {type:"integer"} # Number of columns for arranging components
SPACING_FACTOR_X = 1 # @param {type:"number"} # Horizontal spacing between components (increased for better separation)
SPACING_FACTOR_Y = 1 # @param {type:"number"} # Vertical spacing between components (increased for better separation)
# NODE_COLORMAP = "viridis" # @param ["viridis", "plasma", "inferno", "magma", "cividis", "rainbow", "jet", "hsv", "tab10", "tab20"] {type:"string"} # Colormap for nodes

# --- Configuration for component size scaling ---
BASE_CELL_UNIT = 5 # @param {type:"number"} # Base unit size for component layout. Larger value = larger overall components.
SCALING_EXPONENT = 0.5 # @param {type:"number"} # Exponent for non-linear scaling. 0.0 = uniform size, 1.0 = linear scaling by node count, 0.5 = square root scaling.

# --- New parameter for highlighting characterized nodes ---
HIGHLIGHT_CHARACTERIZED_NODES = True # @param {type:"boolean"}

plt.figure(figsize=(16, 16)) # Adjust figure size for a grid layout

# Get connected components and sort them by size (largest first) for better packing
components = sorted(nx.connected_components(G), key=len, reverse=True)

# Calculate max nodes in any component for scaling, avoid division by zero
if components:
    max_nodes_in_any_component = max(len(c) for c in components)
else:
    max_nodes_in_any_component = 1 # Default to 1 to avoid division by zero

# Dictionary to store final positions for all nodes
final_pos = {}

# Variables for grid positioning
current_x_offset = 0.0
current_y_offset = 0.0
row_max_height = 0.0 # Track max height in current row for y_offset adjustment

current_col = 0

# Store component IDs for node coloring later
comp_id = {}

for i, component_nodes in enumerate(components):
    subgraph = G.subgraph(component_nodes)

    # Calculate current component's target display size based on its node count
    if SCALING_EXPONENT == 1.0: # Uniform size
        current_component_display_size = BASE_CELL_UNIT * 0.5 # A fixed base unit if scaling is off
    elif len(component_nodes) == 1:
        current_component_display_size = BASE_CELL_UNIT * 0.2 # Minimum size for singletons
    else:
        current_component_display_size = BASE_CELL_UNIT * (len(component_nodes) / max_nodes_in_any_component)**SCALING_EXPONENT

    # Apply a layout to the individual component
    # Keep 'k' relatively small for internal compactness, and let external scaling handle the size
    # Using a 'k' proportional to the component's internal spread but not its grid size.
    # Using fruchterman_reingold_layout as it usually gives better separation for subgraphs.
    comp_pos = nx.fruchterman_reingold_layout(subgraph, k=0.001, iterations=50, seed=42 + i)

    # Get bounding box of the layout for internal normalization
    min_x = min(p[0] for p in comp_pos.values())
    max_x = max(p[0] for p in comp_pos.values())
    min_y = min(p[1] for p in comp_pos.values())
    max_y = max(p[1] for p in comp_pos.values())

    # Ensure range is not zero for single-node components or very tight layouts
    x_range_comp = (max_x - min_x) if max_x > min_x else 1.0
    y_range_comp = (max_y - min_y) if max_y > min_y else 1.0

    # Scale the internal layout of the component to fit within its calculated display size
    scaled_comp_pos = {
        node:
            (
                (p[0] - min_x) / x_range_comp * current_component_display_size,
                (p[1] - min_y) / y_range_comp * current_component_display_size,
            )
        for node, p in comp_pos.items()
    }

    # The actual width and height this component will occupy on the grid
    component_actual_width = current_component_display_size
    component_actual_height = current_component_display_size

    # Check if we need to start a new row
    if current_col >= NUM_COLUMNS:
        current_col = 0
        current_y_offset -= (row_max_height + SPACING_FACTOR_Y) # Move down by max height of previous row
        current_x_offset = 0.0
        row_max_height = 0.0 # Reset max height for new row

    # Update max height for the current row
    row_max_height = max(row_max_height, component_actual_height)

    # Add shifted positions to the final_pos dictionary
    for node, (x, y) in scaled_comp_pos.items():
        final_pos[node] = (x + current_x_offset, y + current_y_offset)
        comp_id[node] = i # Store component ID for coloring

    # Update x_offset for the next component in the current row
    current_x_offset += component_actual_width + SPACING_FACTOR_X
    current_col += 1


# --- Drawing the network ---

# Assign node sizes: making all nodes the same size
node_sizes_base = 20 # Base size for all nodes
node_sizes = [node_sizes_base] * len(G.nodes())


# Custom coloring logic based on characterized members in each component
# `main_table` is expected to be available from previous cells.
characterized_genbanks = set(main_table['GenBank'].dropna().tolist())

component_char_counts = {}
for i, comp_nodes in enumerate(components):
    count = 0
    for node in comp_nodes:
        if node in characterized_genbanks:
            count += 1
    component_char_counts[i] = count

node_colors_final = []
max_char_count_overall = max(component_char_counts.values()) if component_char_counts else 0

if max_char_count_overall > 0:
    cmap = plt.colormaps['rainbow'] # User requested 'rainbow'
    norm = plt.Normalize(vmin=1, vmax=max_char_count_overall) # vmin=1 so 0 is distinct

    for node in G.nodes():
        comp_idx = comp_id.get(node, -1) # Get component index for the node
        char_count = component_char_counts.get(comp_idx, 0)

        if char_count == 0:
            node_colors_final.append('black') # Clusters with no characterized members are black
        else:
            node_colors_final.append(cmap(norm(char_count)))
else:
    # If no characterized members in any component, all nodes are black
    node_colors_final = ['black'] * len(G.nodes())
    print("No characterized members found in any component. All nodes will be colored black.")

nx.draw_networkx_edges(G, final_pos, alpha=0.2, width=0.5, edge_color='lightgrey')

# Draw all nodes with their component-based coloring
nx.draw_networkx_nodes(G, final_pos, node_size=node_sizes, node_color=node_colors_final)

highlight_nodes = [] # Initialize for title check
# Highlight characterized nodes if the parameter is True
if HIGHLIGHT_CHARACTERIZED_NODES:
    highlight_nodes = [node for node in G.nodes() if node in characterized_genbanks]
    if highlight_nodes:
        # Draw characterized nodes as yellow stars, larger than other nodes
        nx.draw_networkx_nodes(
            G, final_pos,
            nodelist=highlight_nodes,
            node_size=node_sizes_base * 4, # Make them significantly larger
            node_color='yellow',
            node_shape='*', # Star shape
            edgecolors='black', # Add a black border for definition
            linewidths=0.5, # Border width
            alpha=1.0 # Ensure they are fully visible
        )
        print(f"Highlighted {len(highlight_nodes)} characterized nodes as yellow stars.")
    else:
        print("No characterized nodes found to highlight.")


# Add a colorbar if there are characterized members
if max_char_count_overall > 0:
    # Create a dummy scalar mappable for the colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array(list(range(1, max_char_count_overall + 1)))
    cbar = plt.colorbar(sm, ax=plt.gca(), orientation='vertical', fraction=0.02, pad=0.04)
    cbar.set_label('Number of Characterized Members in Component')

title_text = (f"{GH_FAMILY} SSN (Variable Size Grid, percent-identity>={MIN_PIDENT}%, e-value<={MAX_EVALUE}, bitscore>={MIN_BITS})\n"
              f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
              f"{len(components)} components\n"
              f"Nodes colored by count of characterized members in their component (black if none)")
if HIGHLIGHT_CHARACTERIZED_NODES and highlight_nodes:
    title_text += f"\nCharacterized nodes highlighted as yellow stars ({len(highlight_nodes)} nodes)"
plt.title(title_text)
plt.axis("off")
plt.tight_layout()
plt.savefig(ssn_dir / "ssn_grid_variable_size_char_colored.png", dpi=150) # New filename
plt.show()

# **QUESTIONS**
### 1. How many large clusters do you find ?
### 2. Where are the characterized members ?
### 3. Do you see any large uncharacterized clusters ?



---


# **OPTIONAL:**

## Visualize an interactive SSN in Cytoscape

In [ ]:
# @title Export SSN to GraphML format and open in Cytoscape
from google.colab import files
import networkx as nx

# Ensure ssn_dir and GH_FAMILY are defined from previous cells
# G is also expected to be defined from previous SSN construction steps

output_graphml_path = ssn_dir / f"{GH_FAMILY}_ssn.graphml"

# --- Add node attributes for Cytoscape visualization ---
# Retrieve necessary data from previous cells' execution state
# `main_table` is expected to contain characterized members
# `comp_id` maps nodes to their component index
# `component_char_counts` stores characterized member counts per component index

# 1. Identify characterized nodes
characterized_genbanks = set(main_table['GenBank'].dropna().tolist())

# 2. Iterate through each node in the graph and add attributes
for node_id in G.nodes():
    # Add 'is_characterized' attribute
    is_char = node_id in characterized_genbanks
    G.nodes[node_id]['is_characterized'] = is_char

    # Add 'characterized_members_in_component' attribute
    component_index = comp_id.get(node_id) # Get the component index for the current node
    if component_index is not None:
        char_count_in_component = component_char_counts.get(component_index, 0)
        G.nodes[node_id]['characterized_members_in_component'] = char_count_in_component
    else:
        # Should not happen if comp_id is correctly populated for all nodes in G
        G.nodes[node_id]['characterized_members_in_component'] = 0

# Ensure the GraphML file is created before attempting to download
print(f"Exporting the graph to GraphML: {output_graphml_path}")
nx.write_graphml(G, output_graphml_path)
print("Graph exported successfully with 'is_characterized' and 'characterized_members_in_component' attributes.")

if output_graphml_path.exists():
    print(f"Downloading {output_graphml_path.name}...")
    files.download(output_graphml_path)
else:
    print(f"Error: GraphML file not found at {output_graphml_path} after export attempt.")
    print("Please ensure the SSN creation steps have been executed successfully and 'G' is defined.")